# Topic 3: Great Expectations

> **Phase 7 → Week 15 → Topic 3**

---

## What Is Great Expectations?

Great Expectations (GX) is the most popular open-source data quality framework. Instead of writing custom DQ checks (like notebook 02), GX provides 300+ built-in **Expectations** (assertions), stores results as JSON, and generates beautiful HTML **Data Docs** reports.

---

## Interview Cheat Sheet

**Q: What is Great Expectations and how does it integrate with Spark?**
> Great Expectations is an open-source DQ framework. It validates DataFrames against an **Expectation Suite** — a collection of named assertions (e.g., `expect_column_values_to_not_be_null`). It integrates with Spark via `SparkDFDataset` or the newer **GX Core** API. Results (pass/fail + stats) are stored as JSON **Validation Results**. GX generates HTML **Data Docs** — shareable, browsable DQ reports. Integrates with Airflow: run a `GreatExpectationsOperator` as a pipeline task.

**Q: What is an Expectation Suite?**
> An Expectation Suite is a named collection of Expectations (assertions) for a specific dataset. E.g., `orders.silver.suite` contains all checks for the Silver orders table. Suites are stored as JSON and versioned. You build a suite once (often by profiling a sample dataset), then reuse it across every pipeline run. If a new run's data violates the suite, GX fails the validation.

**Q: What is the difference between a GX Checkpoint and running validate() directly?**
> `validate()` runs expectations against data immediately in code. A **Checkpoint** is a configured, re-runnable validation that: (1) connects to a data source, (2) runs an expectation suite, (3) stores results, (4) generates Data Docs, and (5) can send alerts via Action. Checkpoints are how you integrate GX into CI/CD and production pipelines — define once, run repeatedly.

In [ ]:
# Install: pip install great-expectations
# Check version: great_expectations --version

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Week15 - Great Expectations") \
    .master("local[2]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Sample data
orders = spark.createDataFrame([
    ("O001", "C001",  250.0, "shipped",  "2024-01-15"),
    ("O002", "C001",   80.0, "pending",  "2024-01-15"),
    ("O003", "C002",  180.0, "shipped",  "2024-01-15"),
    ("O004", "C002",  320.0, "cancelled","2024-01-15"),
], ["order_id", "customer_id", "amount", "status", "date"])

print("Great Expectations demo data:")
orders.show()

## Part 1: GX Core API (v1.x)

In [ ]:
print("""
Great Expectations Core API (v1.x) — Pattern:
══════════════════════════════════════════════════════════════

import great_expectations as gx
from great_expectations.datasource.fluent import SparkDatasource

# 1. Create a GX Context
context = gx.get_context()

# 2. Add a Spark datasource
datasource = context.data_sources.add_spark(name="spark_source")

# 3. Add a data asset (a specific DataFrame or table)
data_asset = datasource.add_dataframe_asset(name="silver_orders")

# 4. Create a batch request (the actual data to validate)
batch_request = data_asset.build_batch_request(dataframe=orders_df)

# 5. Create an Expectation Suite
suite = context.add_expectation_suite(expectation_suite_name="silver.orders.suite")

# 6. Create a Validator and add Expectations
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name="silver.orders.suite"
)

# Add expectations
validator.expect_column_values_to_not_be_null("order_id")
validator.expect_column_values_to_not_be_null("customer_id")
validator.expect_column_values_to_be_unique("order_id")
validator.expect_column_values_to_be_in_set("status", ["shipped", "pending", "cancelled"])
validator.expect_column_values_to_be_between("amount", min_value=0, strict_min=True)
validator.expect_table_row_count_to_be_between(min_value=1)

# Save the suite
validator.save_expectation_suite(discard_failed_expectations=False)

# 7. Run validation
results = validator.validate()
print(f"Success: {results.success}")
print(f"Statistics: {results.statistics}")
══════════════════════════════════════════════════════════════
""")

## Part 2: Key Expectations Catalogue

In [ ]:
print("""
Great Expectations — Most Useful Expectations:
══════════════════════════════════════════════════════════════

TABLE-LEVEL:
  expect_table_row_count_to_be_between(min_value=100, max_value=10000)
  expect_table_row_count_to_equal(value=5000)
  expect_table_columns_to_match_ordered_list(column_list=['id','name','email'])
  expect_table_columns_to_match_set(column_set={'id','name','email'})

COMPLETENESS:
  expect_column_values_to_not_be_null(column='customer_id')
  expect_column_values_to_not_be_null(
      column='email',
      mostly=0.99)                          # at least 99% non-null

UNIQUENESS:
  expect_column_values_to_be_unique(column='order_id')
  expect_column_pair_values_to_be_unique(
      column_A='customer_id', column_B='date')  # unique per customer per day

VALIDITY:
  expect_column_values_to_be_in_set(
      column='status',
      value_set=['shipped', 'pending', 'cancelled'])

  expect_column_values_to_match_regex(
      column='email',
      regex=r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$')

  expect_column_values_to_be_between(
      column='amount',
      min_value=0.01, max_value=100000.0)

  expect_column_values_to_be_of_type(
      column='amount', type_='DoubleType')

DISTRIBUTION:
  expect_column_mean_to_be_between(
      column='amount', min_value=100, max_value=500)  # avg order value

  expect_column_median_to_be_between(
      column='amount', min_value=50, max_value=300)

  expect_column_stdev_to_be_between(
      column='amount', min_value=10, max_value=1000)

  expect_column_quantile_values_to_be_between(
      column='amount',
      quantile_ranges={
          'quantiles': [0.25, 0.5, 0.75],
          'value_ranges': [[50,200], [100,400], [200,800]]
      })

TEMPORAL:
  expect_column_values_to_be_between(
      column='event_date',
      min_value='2020-01-01', max_value='2025-12-31')  # plausible date range

  expect_column_values_to_match_strftime_format(
      column='event_date', strftime_format='%Y-%m-%d')

The `mostly` parameter (all column-level expectations):
  mostly=0.95  → 95% of rows must pass (allow 5% deviation)
  mostly=1.0   → all rows must pass (default)
  Use mostly for: soft constraints, known dirty data, gradual quality improvement
══════════════════════════════════════════════════════════════
""")

## Part 3: Manual DQ with GX-style Reporting

In [ ]:
# Simulate GX-style validation results without installing GX
# (so this runs in our Docker environment without extra dependencies)
from dataclasses import dataclass, field
from typing import List
import json

@dataclass
class ExpectationResult:
    expectation_type: str
    success: bool
    column: str
    element_count: int
    unexpected_count: int
    unexpected_percent: float
    kwargs: dict = field(default_factory=dict)

def run_suite(df, suite_name: str, expectations: List[dict]) -> dict:
    total = df.count()
    results = []

    for exp in expectations:
        exp_type = exp["type"]
        col      = exp.get("column")
        mostly   = exp.get("mostly", 1.0)

        if exp_type == "not_null":
            bad = df.filter(F.col(col).isNull()).count()
        elif exp_type == "unique":
            bad = total - df.select(col).distinct().count()
        elif exp_type == "in_set":
            bad = df.filter(~F.col(col).isin(exp["values"]) | F.col(col).isNull()).count()
        elif exp_type == "positive":
            bad = df.filter(F.col(col) <= 0).count()
        elif exp_type == "row_count_min":
            bad = 0 if total >= exp["min"] else 1
            col  = "(table)"
        else:
            bad = 0

        pct     = bad / total if total > 0 else 0
        success = pct <= (1 - mostly)

        results.append(ExpectationResult(
            expectation_type=exp_type,
            success=success,
            column=col,
            element_count=total,
            unexpected_count=bad,
            unexpected_percent=round(pct * 100, 2),
            kwargs=exp
        ))

    passed = sum(1 for r in results if r.success)
    suite_result = {
        "suite_name": suite_name,
        "success":    all(r.success for r in results),
        "statistics": {
            "evaluated":  len(results),
            "successful": passed,
            "failed":     len(results) - passed,
            "success_pct": round(passed / len(results) * 100, 1)
        },
        "results": [
            {"expectation": r.expectation_type, "column": r.column,
             "success": r.success, "unexpected_count": r.unexpected_count,
             "unexpected_percent": r.unexpected_percent}
            for r in results
        ]
    }
    return suite_result

# Define suite
suite = [
    {"type": "row_count_min", "min": 1},
    {"type": "not_null",  "column": "order_id"},
    {"type": "not_null",  "column": "customer_id"},
    {"type": "unique",    "column": "order_id"},
    {"type": "in_set",    "column": "status",
     "values": ["shipped", "pending", "cancelled"]},
    {"type": "positive",  "column": "amount"},
]

# Run on valid data
result = run_suite(orders, "silver.orders.suite", suite)
print("=== Validation Results ===")
print(json.dumps(result, indent=2))

# Show what failure looks like
bad_orders = spark.createDataFrame([
    ("O001", None,  -10.0, "unknown"),
    ("O001", "C1",   50.0, "shipped"),  # duplicate + first has null customer
], ["order_id", "customer_id", "amount", "status"])

print("\n=== Bad Data Validation ===")
bad_result = run_suite(bad_orders, "silver.orders.suite", suite)
print(f"Overall success: {bad_result['success']}")
print(f"Statistics: {bad_result['statistics']}")
for r in bad_result['results']:
    icon = "✅" if r['success'] else "❌"
    print(f"  {icon} {r['expectation']}({r['column']}): {r['unexpected_count']} failures")

## Part 4: GX + Airflow Integration

In [ ]:
print("""
Great Expectations + Airflow Integration:
══════════════════════════════════════════════════════════════

Option 1: GreatExpectationsOperator (airflow-provider-great-expectations)

  from great_expectations_provider.operators.great_expectations import GreatExpectationsOperator

  validate_silver = GreatExpectationsOperator(
      task_id='validate_silver_orders',
      data_context_root_dir='/opt/airflow/great_expectations',  # GX project dir
      checkpoint_name='silver_orders_checkpoint',
      fail_task_on_validation_failure=True,   # DAG fails if any expectation fails
      return_json_dict=True,                  # push results to XCom
  )

  # In DAG:
  silver_write >> validate_silver >> gold_build

Option 2: PythonOperator wrapping GX validation

  def validate_with_gx(**context):
      import great_expectations as gx
      ctx = gx.get_context()

      # Read the data written by the Silver step
      spark = SparkSession.builder.getOrCreate()
      date  = context['ds']
      df    = spark.read.parquet(f's3://bucket/silver/orders/date={date}/')

      datasource  = ctx.data_sources.get('spark_source')
      asset       = datasource.get_asset('silver_orders')
      batch_req   = asset.build_batch_request(dataframe=df)

      results = ctx.run_checkpoint(
          checkpoint_name='silver_orders_checkpoint',
          batch_request=batch_req,
      )

      if not results.success:
          failed = [str(r) for r in results.run_results.values()
                    if not r.success]
          raise ValueError(f"GX validation failed: {failed}")

  gx_validate = PythonOperator(
      task_id='gx_validate_silver',
      python_callable=validate_with_gx,
  )

GX Actions (run automatically on checkpoint):
  StoreValidationResultAction   → save JSON results to S3/local
  UpdateDataDocsAction          → rebuild HTML Data Docs
  SlackNotificationAction       → send Slack alert on failure
  PagerdutyAlertAction          → page on-call on failure
══════════════════════════════════════════════════════════════
""")

## Exercises

1. Define a GX Expectation Suite for a `silver.payments` table with: non-null `payment_id` and `order_id`, unique `payment_id`, `payment_method` in `['card', 'paypal', 'bank_transfer']`, `amount` between 0.01 and 50000, and row count between 1000 and 10M. Write the Python code.
2. What is the `mostly` parameter in GX? Write a suite for a `bronze.events` table where you know the source has ~2% nulls on `session_id` (acceptable) but zero tolerance for null `event_id`. Show how `mostly` handles the session case.
3. Design a GX Checkpoint that: validates `silver.orders`, stores results to `s3://bucket/gx-results/`, updates Data Docs on S3, and sends a Slack alert on any failure. List the Actions configuration.
4. Compare writing custom DQ checks (notebook 02 approach) vs using Great Expectations. List 3 advantages of each. When would you build custom DQ instead of using GX?
5. Your GX suite has 20 expectations. One of them (`expect_column_mean_to_be_between` for `amount`) fails because of a legitimate business change (prices increased across the board). How do you update the expectation without breaking the CI pipeline for unrelated tests?